In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,split,explode,size
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = (
    SparkSession.builder
    .appName("iceberg-learning")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.local.type",
        "hadoop"
    )
    .config(
        "spark.sql.catalog.local.warehouse",
        "/data/warehouse"
    )
    .getOrCreate()
)

df = spark.read.parquet("/data/raw/machine-raw-log-anonymized.parquet").select("ChassisId","Cell","LogId","CellValue","CreationDateTime")

In [19]:
df.columns

['ChassisId', 'Cell', 'LogId', 'CellValue', 'CreationDateTime']

In [20]:
df.show()

+-------------+----+-----+---------+--------------------+
|    ChassisId|Cell|LogId|CellValue|    CreationDateTime|
+-------------+----+-----+---------+--------------------+
|CHS66919BA506|   1| 1061|  2083007|2021-05-24 15:33:...|
|CHS66919BA506|   2| 1061|      577|2021-05-24 15:33:...|
|CHS66919BA506|   3| 1061|      598|2021-05-24 15:33:...|
|CHS66919BA506|   4| 1061|        0|2021-05-24 15:33:...|
|CHS66919BA506|   5| 1061|   536585|2021-05-24 15:33:...|
|CHS66919BA506|   6| 1061|        0|2021-05-24 15:33:...|
|CHS66919BA506|   7| 1061|      960|2021-05-24 15:33:...|
|CHS66919BA506|   1|  324|  2083005|2021-05-24 15:33:...|
|CHS66919BA506|   2|  324|   569190|2021-05-24 15:33:...|
|CHS66919BA506|   3|  324|   141905|2021-05-24 15:33:...|
|CHS66919BA506|   4|  324|   603270|2021-05-24 15:33:...|
|CHS66919BA506|   1|  420|  2083006|2021-05-24 15:33:...|
|CHS66919BA506|   2|  420|        0|2021-05-24 15:33:...|
|CHS66919BA506|   3|  420|        0|2021-05-24 15:33:...|
|CHS66919BA506

In [21]:
mapping_schema = StructType([
    StructField("PIPELINE", StringType(), True),
    StructField("LOG_ID", IntegerType(), True),
    StructField("Cell", StringType(), True),
    StructField("MEASURE_CODE_ACC", StringType(), True),
    StructField("UNIT", DoubleType(), True),
    StructField("TARGET_UNIT", StringType(), True),
    StructField("DESCRIPTION", StringType(), True)
])

df_mapping = spark.read.option("header", "true").schema(mapping_schema).csv("/data/raw/log_cell_mapping.csv")

In [22]:
df_mapping = df_mapping.withColumn("Cell_Array",split(df_mapping['Cell'],",")).withColumn("Cell_number",explode("Cell_Array")).withColumn("Cell_Count",size(col("Cell_Array")))

In [23]:
df_mapping.show()

+------------+------+----+--------------------+-----+-----------+--------------------+----------+-----------+----------+
|    PIPELINE|LOG_ID|Cell|    MEASURE_CODE_ACC| UNIT|TARGET_UNIT|         DESCRIPTION|Cell_Array|Cell_number|Cell_Count|
+------------+------+----+--------------------+-----+-----------+--------------------+----------+-----------+----------+
|      HAULER|  1061|   1|     ENG_HOURS_TOTAL| 0.05|      Hours|Total cumulative ...|       [1]|          1|         1|
|      HAULER|  1061| 1,2|     TOTAL_FUEL_USED|0.001|     Liters|Cumulative engine...|    [1, 2]|          1|         2|
|      HAULER|  1061| 1,2|     TOTAL_FUEL_USED|0.001|     Liters|Cumulative engine...|    [1, 2]|          2|         2|
|      HAULER|  1080|   3|    IDLE_HOURS_TOTAL|  0.1|      Hours|Total engine idle...|       [3]|          3|         1|
|      HAULER|  1080| 4,5|      IDLE_FUEL_USED|0.001|     Liters|Total fuel consum...|    [4, 5]|          4|         2|
|      HAULER|  1080| 4,5|      

In [24]:
lkpDF = df_mapping.select("MEASURE_CODE_ACC","LOG_ID","Cell_number","UNIT","Cell_Array","Cell_Count","PIPELINE")
cond = [((df.LogId == lkpDF.LOG_ID) &\
         (df.Cell == lkpDF.Cell_number))]
mergedDF = df.join(lkpDF,on = cond, how = "inner" )
mergedDF = mergedDF.withColumn("CellValue_new",col("CellValue")*col("UNIT"))

In [25]:
mergedDF.show()

+-------------+----+-----+---------+--------------------+----------------+------+-----------+-----+----------+----------+--------+------------------+
|    ChassisId|Cell|LogId|CellValue|    CreationDateTime|MEASURE_CODE_ACC|LOG_ID|Cell_number| UNIT|Cell_Array|Cell_Count|PIPELINE|     CellValue_new|
+-------------+----+-----+---------+--------------------+----------------+------+-----------+-----+----------+----------+--------+------------------+
|CHS66919BA506|   1| 1061|  2083007|2021-05-24 15:33:...| TOTAL_FUEL_USED|  1061|          1|0.001|    [1, 2]|         2|  HAULER|          2083.007|
|CHS66919BA506|   1| 1061|  2083007|2021-05-24 15:33:...| ENG_HOURS_TOTAL|  1061|          1| 0.05|       [1]|         1|  HAULER|         104150.35|
|CHS66919BA506|   2| 1061|      577|2021-05-24 15:33:...| TOTAL_FUEL_USED|  1061|          2|0.001|    [1, 2]|         2|  HAULER|             0.577|
|CHS86CECE4B6D|   1| 1061|  2336943|2021-05-24 15:43:...| TOTAL_FUEL_USED|  1061|          1|0.001| 

In [26]:
groupbyCols = [
    "ChassisId",
    "CreationDateTime",
    "LogId",
    "MEASURE_CODE_ACC",
    "PIPELINE"
]
mergedDF = mergedDF.groupBy(groupbyCols).agg(F.sum("CellValue_new").alias("MEASURE_CODE_ACC_VALUE"))

In [27]:
mergedDF.show()

+-------------+--------------------+-----+----------------+--------+----------------------+
|    ChassisId|    CreationDateTime|LogId|MEASURE_CODE_ACC|PIPELINE|MEASURE_CODE_ACC_VALUE|
+-------------+--------------------+-----+----------------+--------+----------------------+
|CHS86CECE4B6D|2021-05-25 04:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|    119097.15000000001|
|CHS86CECE4B6D|2021-05-30 23:43:...| 1061| TOTAL_FUEL_USED|  HAULER|              2812.403|
|CHS66919BA506|2021-05-29 22:33:...| 1061| ENG_HOURS_TOTAL|  HAULER|    123625.40000000001|
|CHS66919BA506|2021-05-30 02:03:...| 1061| ENG_HOURS_TOTAL|  HAULER|    124255.45000000001|
|CHS86CECE4B6D|2021-05-29 15:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|             137831.45|
|CHS86CECE4B6D|2021-05-29 17:13:...| 1061| TOTAL_FUEL_USED|  HAULER|              2764.791|
|CHS86CECE4B6D|2021-05-27 17:13:...| 1061| TOTAL_FUEL_USED|  HAULER|    2592.0510000000004|
|CHS86CECE4B6D|2021-05-28 02:13:...| 1061| TOTAL_FUEL_USED|  HAULER|            

In [29]:
mergedDF.writeTo("local.db.merged_logs").createOrReplace()

In [30]:
spark.sql("USE local.db")
spark.sql("SELECT * FROM merged_logs LIMIT 5").show()

+-------------+--------------------+-----+----------------+--------+----------------------+
|    ChassisId|    CreationDateTime|LogId|MEASURE_CODE_ACC|PIPELINE|MEASURE_CODE_ACC_VALUE|
+-------------+--------------------+-----+----------------+--------+----------------------+
|CHS86CECE4B6D|2021-05-25 04:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|    119097.15000000001|
|CHS86CECE4B6D|2021-05-30 23:43:...| 1061| TOTAL_FUEL_USED|  HAULER|              2812.403|
|CHS66919BA506|2021-05-29 22:33:...| 1061| ENG_HOURS_TOTAL|  HAULER|    123625.40000000001|
|CHS66919BA506|2021-05-30 02:03:...| 1061| ENG_HOURS_TOTAL|  HAULER|    124255.45000000001|
|CHS86CECE4B6D|2021-05-29 15:13:...| 1061| ENG_HOURS_TOTAL|  HAULER|             137831.45|
+-------------+--------------------+-----+----------------+--------+----------------------+



In [31]:
spark.sql("""
    UPDATE local.db.merged_logs
    SET MEASURE_CODE_ACC_VALUE = 999.0
    WHERE ChassisId = 'H100E121013'
    AND MEASURE_CODE_ACC = 'ENG_HOURS_TOTAL'
""")

DataFrame[]

In [32]:
spark.sql("""
    DELETE FROM local.db.merged_logs
    WHERE LogId = 1061
    AND ChassisID = 'H100E121013'
""")

DataFrame[]

In [33]:
spark.sql("SELECT * FROM local.db.merged_logs.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                               |summary                                                                                                                                                                                                                                                        

In [34]:
spark.sql("SHOW TBLPROPERTIES local.db.merged_logs").show(truncate=False)

+-------------------------------+-------------------+
|key                            |value              |
+-------------------------------+-------------------+
|current-snapshot-id            |8846844415118150805|
|format                         |iceberg/parquet    |
|format-version                 |2                  |
|write.parquet.compression-codec|zstd               |
+-------------------------------+-------------------+

